# Chapter 1: Why LLMs Matter for BA & QA

**Book:** [LLMs for Business & Quality Analysts](../index.html)

## Lab Overview
Build an **LLM Impact Assessment Tool** that evaluates how LLMs can transform specific BA/QA tasks in your organization. You will score tasks on automation potential, quality improvement, and time savings.

## Prerequisites
- Python 3.9+
- OpenAI API key
- Basic understanding of BA/QA workflows


In [ ]:
# Setup
!pip install -q openai python-dotenv pandas

import os
import json
from openai import OpenAI
from dotenv import load_dotenv
import pandas as pd

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o'

## 1. Define BA/QA Tasks to Assess

We start by listing common BA and QA tasks, then ask the LLM to evaluate each one for automation potential.


In [ ]:
# Define BA/QA tasks for impact assessment
tasks = [
    'Requirements gathering from stakeholder interviews',
    'Writing user stories with acceptance criteria',
    'Creating process flow diagrams from text descriptions',
    'Generating test cases from requirements',
    'Writing test data for edge cases',
    'Triaging and categorizing defect reports',
    'Creating sprint retrospective summaries',
    'Reviewing BRD documents for completeness',
]

print(f'Assessing {len(tasks)} BA/QA tasks for LLM impact...')

## 2. LLM Impact Assessment Engine

We send each task to the LLM and ask it to rate automation potential, quality improvement, and time savings on a 1-10 scale.


In [ ]:
def assess_task(task: str) -> dict:
    """Use an LLM to assess the impact of AI on a specific BA/QA task."""
    prompt = f"""You are an AI transformation consultant. Assess the following BA/QA task 
for LLM automation potential. Return a JSON object with these fields:
- task: the task name
- automation_potential: score 1-10
- quality_improvement: score 1-10  
- time_savings_percent: estimated percentage of time saved (0-90)
- recommended_approach: one of 'full_automation', 'human_in_the_loop', 'ai_assisted'
- rationale: one sentence explanation

Task: {task}

Return ONLY valid JSON, no markdown."""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.3
    )
    return json.loads(response.choices[0].message.content)

# Assess all tasks
results = []
for task in tasks:
    result = assess_task(task)
    results.append(result)
    print(f'Assessed: {task[:50]}... -> Automation: {result["automation_potential"]}/10')

df = pd.DataFrame(results)
df

In [ ]:
# Generate a summary report
summary_prompt = f"""Based on these LLM impact assessment results for BA/QA tasks, 
write a concise executive summary (3-4 paragraphs) with recommendations:

{df.to_string()}

Include: top opportunities, risks to watch, and a suggested adoption roadmap."""

response = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': summary_prompt}],
    temperature=0.5
)
print(response.choices[0].message.content)

## Exercises
1. Add 5 more BA/QA tasks specific to your organization and re-run the assessment.
2. Modify the prompt to include a `risk_level` field (low/medium/high) for each task.
3. Create a bar chart visualization of the automation potential scores using matplotlib.
4. Compare results using different temperature values (0.1 vs 0.7) -- how do scores change?
